In [1]:
# Prototype for dashboard computations

# Performance
# ytd perf

# Risk
# 20-day historical vol
# 1-day VaR

# Just storing stuff in the DB. Only portfolio level for now
# First compute portfolio values

In [1]:
import datetime
from sqlalchemy import and_, func, select
from sqlalchemy.orm import Session

from worker.database import Holding, Portfolio
from worker.database import MarketData, connect
import datetime
import polars as pl

from sqlalchemy.dialects.postgresql import insert

from worker.database import PortfolioValue


def compute_portfolio_values(db: Session, date: datetime.date):
    # Subquery to find the max as_of date for each portfolio-instrument combo
    # up to the target date
    max_date_subq = (
        select(Holding.portfolio_id, Holding.instrument_id, func.max(Holding.as_of).label("max_as_of"))
        .where(Holding.as_of <= date)
        .group_by(Holding.portfolio_id, Holding.instrument_id)
        .subquery()
    )

    holdings = (
        db.execute(
            select(Holding).join(
                max_date_subq,
                and_(
                    Holding.portfolio_id == max_date_subq.c.portfolio_id,
                    Holding.instrument_id == max_date_subq.c.instrument_id,
                    Holding.as_of == max_date_subq.c.max_as_of,
                ),
            )
        )
        .scalars()
        .all()
    )

    return holdings

In [47]:
dr = pl.date_range(datetime.date(2000, 1, 1), datetime.date(2025, 8, 28), eager=True)
dr_bus = dr.filter(dr.dt.is_business_day()).to_list()
df_dates = pl.DataFrame({"date": dr_bus})
print(len(dr_bus))

6694


In [ ]:
# Propagate holdings

with connect() as session:
    df_holdings = pl.read_database(
        """\
        SELECT * FROM holdings
        """,
        session,
        # execute_options={"parameters": {"date": date}}
    )

    df_result = df_dates.join(df_holdings, how="cross").with_columns(as_of=pl.col("date")).drop("date")

    df_result.write_database("holdings", session, if_table_exists="replace")

    session.commit()


In [45]:
def _prev_bus_day(date: datetime.date):
    return (
        pl.select(
            pl.lit(date)
            .dt.add_business_days(-1, roll="forward")
        )
        .to_series()
        .item()
    )


def propagate_holdings(session, date: datetime.date):
    df_latest_holdings = pl.read_database(
        """\
        SELECT holdings.portfolio_id, holdings.instrument_id, holdings.quantity, T.max_date
        FROM holdings
        INNER JOIN (
            SELECT portfolio_id, MAX(as_of) as max_date
            FROM holdings
            GROUP BY portfolio_id
        ) AS T
        ON holdings.portfolio_id = T.portfolio_id
        AND holdings.as_of = T.max_date
        
        """,
        session,
        # execute_options={"parameters": {"date": date}}
    )

    # Check
    prev_date = _prev_bus_day(date)
    all_max_dates = df_latest_holdings["max_date"].unique().to_list()

    assert len(all_max_dates) == 1
    assert all_max_dates[0] == prev_date

    return df_latest_holdings

In [ ]:
with connect() as session:
    df = propagate_holdings(session, datetime.date)

AssertionError: 

In [40]:
_prev_bus_day(datetime.date(2025, 8, 30))

datetime.date(2025, 8, 29)

In [ ]:
for date in dr_bus:
    print(f"Processing {date}...")

    with connect() as session:
        df_values = pl.read_database(
            """\
            SELECT 
                holdings.portfolio_id, 
                holdings.instrument_id, 
                holdings.quantity,
                market_data.value AS adj_close
            FROM holdings
            LEFT JOIN (
                SELECT portfolio_id, MAX(as_of) AS max_date
                FROM holdings
                WHERE as_of <= :date
                GROUP BY portfolio_id        
            ) AS T
            ON holdings.portfolio_id = T.portfolio_id
            AND holdings.as_of = T.max_date

            LEFT JOIN market_data
            ON market_data.date = :date
            AND market_data.instrument_id = holdings.instrument_id

            WHERE market_data.data_type = 'adj_close'

            
            """,
            session,
            execute_options={"parameters": {"date": date}},
        )

    df_ptf_values = (
        df_values.with_columns(value=pl.col("quantity") * pl.col("adj_close"))
        .group_by("portfolio_id")
        .agg(value=pl.col("value").sum())
        .with_columns(date=date)
    )

    items_to_add = df_ptf_values.to_dicts()

    chunk_size = 1000
    total_inserted = 0

    for i in range(0, len(items_to_add), chunk_size):
        chunk = items_to_add[i : i + chunk_size]

        try:
            stmt = insert(PortfolioValue).values(chunk)
            stmt = stmt.on_conflict_do_nothing()
            result = session.execute(stmt)
            session.commit()

            chunk_inserted = result.rowcount
            total_inserted += chunk_inserted

            print(f"Chunk {i // chunk_size + 1}: Inserted {chunk_inserted}/{len(chunk)} rows (Total: {total_inserted})")

        except Exception as e:
            print(f"Error inserting chunk {i // chunk_size + 1}: {e}")
            session.rollback()

Processing 2021-01-01...
Processing 2021-01-04...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-05...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-06...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-07...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-08...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-11...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-12...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-13...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-14...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-15...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-18...
Processing 2021-01-19...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-20...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-21...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-22...
Chunk 1: Inserted 50/50 rows (Total: 50)
Processing 2021-01-25...
C

In [ ]:
df_ptf_values = (
    df_values.with_columns(value=pl.col("quantity") * pl.col("adj_close"))
    .group_by("portfolio_id")
    .agg(value=pl.col("value").sum())
    .with_columns(date=date)
)

items_to_add = df_ptf_values.to_dicts()


chunk_size = 1000
total_inserted = 0

with connect() as session:
    for i in range(0, len(items_to_add), chunk_size):
        chunk = items_to_add[i : i + chunk_size]

        try:
            stmt = insert(PortfolioValue).values(chunk)
            stmt = stmt.on_conflict_do_nothing()
            result = session.execute(stmt)
            session.commit()

            chunk_inserted = result.rowcount
            total_inserted += chunk_inserted

            print(f"Chunk {i // chunk_size + 1}: Inserted {chunk_inserted}/{len(chunk)} rows (Total: {total_inserted})")

        except Exception as e:
            print(f"Error inserting chunk {i // chunk_size + 1}: {e}")
            session.rollback()

Chunk 1: Inserted 50/50 rows (Total: 50)


In [27]:
df_max_dates

portfolio_id,max
object,date
65bd0c90-941b-4ff5-bafd-30ab87c5f539,2025-08-29
2b703602-f3f0-42a1-97f7-cb7e9a0ab92b,2025-08-29
2eadb74c-b1e5-4763-8c10-e4c5c4a7a483,2025-08-29
cb90f820-ba51-4ddf-8bbf-d54730cc3482,2025-08-29
1e28aa46-048c-42e1-a098-49cf4de0ebe3,2025-08-29
…,…
15cf150e-4108-4f5d-85f0-ddf047128e1d,2025-08-29
d941b6be-8ce3-4b06-9e93-fa105153a697,2025-08-29
502f0588-e1f3-4df9-b202-87c6e8af26ec,2025-08-29


In [19]:
holdings

In [13]:
portfolios_dict

{UUID('bd3425c7-1dea-44bd-b345-d7517316f314'): {'portfolio': <worker.database.Portfolio at 0x11780fa10>,
  'holdings': [<worker.database.Holding at 0x116f56040>]},
 UUID('30ecbfb2-2fa9-4611-afcf-8b8fd61aa429'): {'portfolio': <worker.database.Portfolio at 0x11780fa80>,
  'holdings': [<worker.database.Holding at 0x116f54750>]},
 UUID('7509fde2-c757-486d-9b53-90f28b5d5a0c'): {'portfolio': <worker.database.Portfolio at 0x11780faf0>,
  'holdings': [<worker.database.Holding at 0x116f546e0>]},
 UUID('6c5d5b6e-f50a-40fe-ba5e-b208e1ecb458'): {'portfolio': <worker.database.Portfolio at 0x11780cde0>,
  'holdings': [<worker.database.Holding at 0x116f55cc0>]},
 UUID('24d4e3a0-5ea6-4ace-b18a-8ac9a16fcb8e'): {'portfolio': <worker.database.Portfolio at 0x11780dbe0>,
  'holdings': [<worker.database.Holding at 0x116f54670>]},
 UUID('2b8e2f4b-c492-4521-89f3-0a4f8a2afcb1'): {'portfolio': <worker.database.Portfolio at 0x11780d240>,
  'holdings': [<worker.database.Holding at 0x116f55c50>]},
 UUID('6528a25b-

In [7]:
portfolios_dict

{UUID('bd3425c7-1dea-44bd-b345-d7517316f314'): {'portfolio': <worker.database.Portfolio at 0x116f11010>,
  'holdings': [<worker.database.Holding at 0x1169875b0>]},
 UUID('30ecbfb2-2fa9-4611-afcf-8b8fd61aa429'): {'portfolio': <worker.database.Portfolio at 0x116eede50>,
  'holdings': [<worker.database.Holding at 0x116f04c20>]},
 UUID('7509fde2-c757-486d-9b53-90f28b5d5a0c'): {'portfolio': <worker.database.Portfolio at 0x116eedd10>,
  'holdings': [<worker.database.Holding at 0x116f052b0>]},
 UUID('6c5d5b6e-f50a-40fe-ba5e-b208e1ecb458'): {'portfolio': <worker.database.Portfolio at 0x116e7a520>,
  'holdings': [<worker.database.Holding at 0x116f05240>]},
 UUID('24d4e3a0-5ea6-4ace-b18a-8ac9a16fcb8e'): {'portfolio': <worker.database.Portfolio at 0x116e7a2c0>,
  'holdings': [<worker.database.Holding at 0x116f05320>]},
 UUID('2b8e2f4b-c492-4521-89f3-0a4f8a2afcb1'): {'portfolio': <worker.database.Portfolio at 0x116f34cb0>,
  'holdings': [<worker.database.Holding at 0x116f05390>]},
 UUID('6528a25b-